# 02 — Treino e Validação (YOLOv8n)

**CPZÃO · Auditoria de Calçadas**

* Dataset: 100 imagens (70 train / 20 val / 10 test), 3 classes
* Modelo: **YOLOv8n** (leve, compatível com exportação mobile)
* Métricas: mAP@50, mAP@50-95, Precision, Recall, matriz de confusão e curvas de perda

> No Colab: **Runtime → Change runtime type → GPU (T4)** antes de rodar.

In [ ]:
# checa ambiente e se o dataset ja existe (tem que ter rodado o notebook 01 antes)
import sys, subprocess, glob, os
if 'google.colab' in sys.modules:
    subprocess.run('pip -q install ultralytics'.split(), check=True)
    # cloude/suba o projeto e entre na raiz do repositório
    # !git clone <URL_DO_SEU_REPOSITORIO>.git projeto && %cd projeto
assert os.path.exists('data/dataset/dataset.yaml'), 'dataset não encontrado — rode o notebook 01'
import torch, ultralytics
print('device  :', 'CUDA' if torch.cuda.is_available() else 'CPU')
print('ultralytics', ultralytics.__version__)

In [ ]:
# treino propriamente dito: parte do YOLOv8n pre-treinado e ajusta pras nossas 3 classes
from ultralytics import YOLO
data = 'data/dataset/dataset.yaml'
model = YOLO('yolov8n.pt')          # pesos pré-treinados (COCO)
results = model.train(
    data=data, imgsz=320, epochs=80, batch=16,
    patience=30, cos_lr=True, lr0=0.01,   # para cedo se estagnar; lr em cosseno
    project='runs', name='calcada_v1', exist_ok=True,
    device=0 if torch.cuda.is_available() else 'cpu',  # usa GPU se tiver, senao CPU
    workers=0 if 'google.colab' not in sys.modules else 2,
    seed=0, plots=True,                   # seed fixa p/ reprodutibilidade; plots=True gera as curvas
)
print('melhores pesos:', results.save_dir + '/weights/best.pt')

### Curvas de treino (perdas, mAP e matriz de confusão)

In [ ]:
# exibe as figuras que o treino gerou: curvas de perda/mAP, matriz de confusao e um lote de predicoes
from IPython.display import Image, display, clear_output
d = 'runs/calcada_v1'
display(Image(f'{d}/results.png', width=950))
display(Image(f'{d}/confusion_matrix_normalized.png', width=520))
display(Image(f'{d}/val_batch0_pred.jpg', width=900))

### Avaliação nos splits `val` e `test`

In [ ]:
# roda a validacao no melhor peso e imprime as metricas por split (geral e por classe)
model = YOLO('runs/calcada_v1/weights/best.pt')
rows = {}
for split in ('val', 'test'):
    r = model.val(data=data, split=split, imgsz=320, plots=True, name=split)
    rows[split] = dict(mAP50=r.box.map50, mAP50_95=r.box.map,
                       precision=r.box.mp, recall=r.box.mr)
    per = {name: round(float(v), 4) for name, v in zip(r.names.values(), r.box.map50_class)}
    print(split, '-> mAP50 =', round(r.box.map50, 4), '| por classe:', per)

### Avaliação na resolução de execução no celular (192 px)

In [ ]:
# mede a performance na mesma resolucao em que o modelo vai rodar no smartphone (192)
r = model.val(data=data, split='test', imgsz=192, plots=False, name='test192')
print('test@192 -> mAP50 =', round(r.box.map50, 4), '| mAP50-95 =', round(r.box.map, 4))

### Análise das métricas
* **mAP@50** resume a qualidade geral da detecção (IoU ≥ 0,5);
* **mAP@50-95** penaliza caixas imprecisas — o número que revela a qualidade da anotação;
* **Precision × Recall** indicam o equilíbrio entre falsos positivos e falsos negativos (um
  modelo com Recall baixo "deixa passar" defeitos na auditoria);
* a comparação `test@320` × `test@192` quantifica a perda de rodar no smartphone.